In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [1]:
import os
import gc
import torch
from pathlib import Path

from harreman_funcs import HarremanRunner
import harreman_summary

In [2]:
from metab_processing.metab_travlr_config import PROJECT_DATA_DIR, DATA_DIR as METAB_DATA_DIR

XENIUM_DATA_DIR = PROJECT_DATA_DIR
TIERS = ['Tier1', 'Tier2', 'Tier3']

In [3]:
def run_dataset(data_dir, dataset_name):
    harRunner = HarremanRunner(f'{data_dir}/{dataset_name}')
    harRunner.load_adata()
    harRunner.save_harreman_network()

    tiers = [tier for tier in TIERS if tier in harRunner.adata.obs.columns]
    if not tiers:
        raise ValueError('no tier annotations')

    for tier in tiers:
        harRunner.run_harreman(tier)

    out_path = harRunner.easy_download_path
    master, genepairs = harreman_summary.summarize_harreman_folder(out_path, sample_id=dataset_name)
    summary_dir = Path(out_path) / 'summary'
    summary_dir.mkdir(parents=True, exist_ok=True)
    master.to_csv(summary_dir / 'metabolite_summary.csv', index=False)
    genepairs.to_csv(summary_dir / 'gene_pair_summary.csv', index=False)
    harreman_summary.select_tcell_metabolites(out_path)

    # written last so a dataset is only skipped once it finished
    marker_path(data_dir, dataset_name).write_text(dataset_name)

In [ ]:
def marker_path(data_dir, dataset_name):
    return Path(f'{data_dir}/{dataset_name}/easy_download/.{dataset_name}')


def run_all(data_dir=XENIUM_DATA_DIR):
    d_sets = os.listdir(data_dir)
    # d_sets = ['Primary_Dermal_Melanoma']
    print(d_sets)
    for dataset_name in sorted(d_sets):
        if not os.path.isdir(f'{data_dir}/{dataset_name}'):
            continue
        if marker_path(data_dir, dataset_name).is_file():
            print(f'skipping {dataset_name}, already done')
            continue

        print(f'running {dataset_name}')
        try:
            run_dataset(data_dir, dataset_name)
            print(f'finished {dataset_name}')
        except Exception as e:
            print(f'failed {dataset_name}: {type(e).__name__}: {e}')

        gc.collect()
        torch.cuda.empty_cache()

In [5]:
run_all()

['Human_Breast', 'Human_Lung', 'Human_Prostate_Adenocarcinoma', 'Human_Cervical_Cancer', 'Primary_Dermal_Melanoma']
running Human_Breast
running cell independent
Extracting interaction database...
Finished extracting interaction database in 0.237 seconds
Applying gene filtering...
Finished applying gene filtering in 0.000 seconds
Computing the neighborhood graph...
Computing the weights...
Finished computing the KNN graph in 5.920 seconds
Computing gene pairs...
Finished computing gene pairs in 0.080 seconds
Gene pairs to test: 429
Starting cell-cell communication analysis...
Running the parametric test...
Parametric test finished.
Running the non-parametric test...


Permutation test:   0%|          | 0/1000 [00:00<?, ?it/s]


failed Human_Breast: OutOfMemoryError: CUDA out of memory. Tried to allocate 2.24 GiB. GPU 0 has a total capacity of 23.55 GiB of which 867.25 MiB is free. Including non-PyTorch memory, this process has 22.69 GiB memory in use. Of the allocated memory 20.73 GiB is allocated by PyTorch, and 1.73 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
running Human_Cervical_Cancer
running cell independent
Extracting interaction database...
Finished extracting interaction database in 0.238 seconds
Applying gene filtering...
Finished applying gene filtering in 0.000 seconds
Computing the neighborhood graph...
Computing the weights...
Finished computing the KNN graph in 5.940 seconds
Computing gene pairs...
Finished computing ge

Permutation test: 100%|██████████| 1000/1000 [03:33<00:00,  4.68it/s]


Non-parametric test finished.
Obtaining communication results...
Finished computing cell-cell communication analysis in 216.738 seconds
computing neighborhood scores
[lowmem] Computing gene pair and metabolite scores...
[lowmem] Running the non-parametric test...


...

Non-parametric test finished.
Obtaining cell type-aware communication results...
Finished computing cell type-aware cell-cell communication analysis in 62.763 seconds
Already have ccc_results
loading cell indep results for filtering
Computing gene pairs...
Finished computing gene pairs in 0.290 seconds
Starting cell type-aware cell-cell communication analysis...
Running the parametric test...
Running the non-parametric test...


Permutation test: 100%|██████████| 1000/1000 [01:00<00:00, 16.63it/s]


Non-parametric test finished.
Obtaining cell type-aware communication results...
Finished computing cell type-aware cell-cell communication analysis in 62.436 seconds
Already have ccc_results
loading cell indep results for filtering
Computing gene pairs...
Finished computing gene pairs in 0.449 seconds
Starting cell type-aware cell-cell communication analysis...
Running the parametric test...
Running the non-parametric test...


Permutation test: 100%|██████████| 1000/1000 [01:29<00:00, 11.12it/s]


Non-parametric test finished.
Obtaining cell type-aware communication results...
Finished computing cell type-aware cell-cell communication analysis in 92.309 seconds
Wrote 41 metabolites -> /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Prostate_Adenocarcinoma/easy_download/harreman_outputs/metabolite_selection.yaml
finished Human_Prostate_Adenocarcinoma
skipping Primary_Dermal_Melanoma, already done


In [7]:
from copy_easy_download import save_easy_downloads
save_easy_downloads()